# QualCheck: BERT-Based Rubric-Guided Semantic Evaluation Pipeline for Code Reports

**Camarines Sur Polytechnic Colleges – College of Computer Studies**

This notebook implements the full QualCheck training and evaluation pipeline as described in the Methodology (Chapter 3), specifically customized for evaluating **Code Reports**.

**Dataset Context:** 
A synthetic dataset of 10,000 code report entries encompassing programming assignments and documentation evaluations based on:
- Conceptual understanding
- Explanation clarity
- Methodological correctness
- Technical coherence

**Implementation Pipeline:**
- Phase 1 – Data Collection & Preprocessing
- Phase 2 – BERT Encoding & Fine-Tuning (BERT-base-uncased)
- Phase 3 – Cosine Similarity Scoring 
- Phase 4 – Threshold Calibration and Relevance Classification (Grid Search)
- Phase 5 – System Evaluation + Baseline Comparisons + Statistical Tests

> **Dataset CSV columns expected:** `question`, `evaluation_criterion`, `rubric_descriptor`, `quality_label`, `response`  
> **Labels:** `Fully Relevant`, `Partially Relevant`, `Irrelevant`

In [ ]:
# Install required packages
%pip install transformers torch scikit-learn pandas numpy nltk spacy scipy statsmodels matplotlib seaborn tqdm -q
!python -m spacy download en_core_web_sm -q

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print('✓ All packages installed')

In [ ]:
import os, json, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (BertTokenizer, BertModel,
                          get_linear_schedule_with_warmup)
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score,
                              precision_score, recall_score)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as tfidf_cos

from scipy.stats import pearsonr, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy

print('✓ Imports complete')


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CONFIGURATION  ← adjust paths and column names to match your CSV
# ─────────────────────────────────────────────────────────────────
class Config:
    # ── Data ──────────────────────────────────────────────────────
    DATA_PATH          = "datasets/codereports/qualcheck_dataset.csv"   # ← Code reports dataset

    # Column names (case-sensitive – must match CSV headers exactly)
    COL_QUESTION       = "question"
    COL_CRITERION      = "evaluation_criterion"    # set None if absent
    COL_RUBRIC         = "rubric_descriptor"
    COL_LABEL          = "quality_label"
    COL_RESPONSE       = "response"
    COL_OUTPUT_TYPE    = "output_type"             # set None if absent

    # ── Labels ────────────────────────────────────────────────────
    LABEL2ID   = {"Fully Relevant": 2, "Partially Relevant": 1, "Irrelevant": 0}
    ID2LABEL   = {2: "Fully Relevant",  1: "Partially Relevant", 0: "Irrelevant"}
    NUM_LABELS = 3

    # ── BERT ──────────────────────────────────────────────────────
    BERT_MODEL = "bert-base-uncased"
    MAX_LEN    = 256          # reduce to 128 if GPU memory is tight

    # ── Training ──────────────────────────────────────────────────
    BATCH_SIZE    = 16        # reduce to 8 on low-VRAM GPU
    LR            = 2e-5
    MAX_EPOCHS    = 5
    PATIENCE      = 3         # early stopping
    WARMUP_RATIO  = 0.1
    DROPOUT       = 0.3

    # ── Split ─────────────────────────────────────────────────────
    TRAIN_RATIO   = 0.70
    VAL_RATIO     = 0.15
    TEST_RATIO    = 0.15
    SEED          = 42

    # ── Threshold calibration ─────────────────────────────────────
    THRESH_STEP   = 0.01

    # ── Paths ─────────────────────────────────────────────────────
    OUTPUT_DIR       = "qualcheck_output"
    BEST_CKPT        = "qualcheck_output/best_model.pt"
    FINAL_MODEL      = "qualcheck_output/qualcheck_final.pt"

    # ── Device ────────────────────────────────────────────────────
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# Reproducibility
random.seed(cfg.SEED); np.random.seed(cfg.SEED); torch.manual_seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.SEED)

print(f"Device  : {cfg.DEVICE}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

## Phase 1 – Data Loading & Exploration

In [ ]:
df = pd.read_csv(cfg.DATA_PATH)
print(f"Shape   : {df.shape}")
print(f"Columns : {df.columns.tolist()}\n")

# ── Basic validation ──────────────────────────────────────────────
required = [cfg.COL_QUESTION, cfg.COL_RUBRIC, cfg.COL_LABEL, cfg.COL_RESPONSE]
missing  = [c for c in required if c not in df.columns]
assert not missing, f"Missing columns: {missing}"

# ── Label check ───────────────────────────────────────────────────
unknown = set(df[cfg.COL_LABEL].unique()) - set(cfg.LABEL2ID.keys())
if unknown:
    print(f"⚠ Unknown labels found (will be dropped): {unknown}")
    df = df[df[cfg.COL_LABEL].isin(cfg.LABEL2ID)]

df["label_id"] = df[cfg.COL_LABEL].map(cfg.LABEL2ID)

print("Label distribution:")
print(df[cfg.COL_LABEL].value_counts().to_string())

print("\nSample entry:")
print(df[[cfg.COL_QUESTION, cfg.COL_RUBRIC, cfg.COL_RESPONSE, cfg.COL_LABEL]].iloc[0].to_string())


## Preprocessing – NLTK + spaCy

In [ ]:
class TextPreprocessor:
    """Light preprocessing that preserves semantic content for BERT."""

    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words("english"))
        try:
            self.nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
        except Exception:
            self.nlp = None

    def clean_for_bert(self, text: str) -> str:
        """Minimal clean – keep casing and punctuation for BERT."""
        if pd.isna(text):
            return ""
        return str(text).strip()

    def deep_clean(self, text: str) -> str:
        """Full pipeline: lower → tokenise → remove stops → lemmatise."""
        if pd.isna(text):
            return ""
        text = str(text).lower()
        tokens = nltk.word_tokenize(text)
        tokens = [self.lemmatizer.lemmatize(t) for t in tokens
                  if t.isalnum() and t not in self.stop_words]
        return " ".join(tokens)

prep = TextPreprocessor()

# For BERT encoding – minimal cleaning
df["clean_question"]  = df[cfg.COL_QUESTION].apply(prep.clean_for_bert)
df["clean_rubric"]    = df[cfg.COL_RUBRIC].apply(prep.clean_for_bert)
df["clean_response"]  = df[cfg.COL_RESPONSE].apply(prep.clean_for_bert)

# Combine criterion + rubric → full rubric string sent to BERT
if cfg.COL_CRITERION and cfg.COL_CRITERION in df.columns:
    df["full_rubric"] = (df[cfg.COL_CRITERION].apply(prep.clean_for_bert)
                          + ". " + df["clean_rubric"])
else:
    df["full_rubric"] = df["clean_rubric"]

# Deep-cleaned text for TF-IDF baseline
df["tfidf_question"] = df[cfg.COL_QUESTION].apply(prep.deep_clean)
df["tfidf_rubric"]   = df["full_rubric"].apply(prep.deep_clean)
df["tfidf_response"] = df[cfg.COL_RESPONSE].apply(prep.deep_clean)

print(f"✓ Preprocessing complete  –  {len(df)} rows")
print("\nExample preprocessed question:")
print(df["clean_question"].iloc[0])


In [ ]:
# Stratified 70 / 15 / 15 split
train_df, temp_df = train_test_split(
    df, test_size=(cfg.VAL_RATIO + cfg.TEST_RATIO),
    random_state=cfg.SEED, stratify=df["label_id"])

val_df, test_df = train_test_split(
    temp_df, test_size=0.5,
    random_state=cfg.SEED, stratify=temp_df["label_id"])

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

N = len(df)
print(f"Train : {len(train_df):>6,}  ({len(train_df)/N*100:.1f}%)")
print(f"Val   : {len(val_df):>6,}  ({len(val_df)/N*100:.1f}%)")
print(f"Test  : {len(test_df):>6,}  ({len(test_df)/N*100:.1f}%)")

# Verify stratification
for split, sdf in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"\n{split} label distribution:")
    print(sdf[cfg.COL_LABEL].value_counts().to_string())


## Phase 2 – BERT Encoding & Fine-Tuning

### Dataset Class

In [ ]:
class QualCheckDataset(Dataset):
    """
    Each sample encodes two segments:
      P  →  [CLS] question [SEP] rubric [SEP]
      R  →  [CLS] response [SEP]
    """

    def __init__(self, dataframe, tokenizer, max_len):
        self.df        = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.df)

    def _encode(self, text_a, text_b=None):
        return self.tokenizer.encode_plus(
            text_a, text_b,
            add_special_tokens  = True,
            max_length          = self.max_len,
            padding             = "max_length",
            truncation          = True,
            return_attention_mask = True,
            return_tensors      = "pt"
        )

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        pr = self._encode(row["clean_question"], row["full_rubric"])
        r  = self._encode(row["clean_response"])

        return {
            "pr_ids"  : pr["input_ids"].squeeze(0),
            "pr_mask" : pr["attention_mask"].squeeze(0),
            "r_ids"   : r["input_ids"].squeeze(0),
            "r_mask"  : r["attention_mask"].squeeze(0),
            "label"   : torch.tensor(row["label_id"], dtype=torch.long)
        }


### QualCheck Siamese-BERT Model

In [ ]:
class QualCheckModel(nn.Module):
    """
    Siamese BERT:
      • Encodes P and R separately with shared BERT weights
      • Computes cosine similarity for inference
      • Trains a lightweight classification head on [P·R, |P−R|, P, R]
        so BERT representations align with relevance labels
    """

    def __init__(self, bert_name, num_labels=3, dropout=0.3):
        super().__init__()
        self.bert       = BertModel.from_pretrained(bert_name)
        hidden          = self.bert.config.hidden_size   # 768

        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 4, hidden),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_labels)
        )

    # ── Shared encoder ────────────────────────────────────────────
    def encode(self, input_ids, attention_mask):
        """Returns [CLS] embedding (768-dim)."""
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return self.dropout(out.last_hidden_state[:, 0, :])   # [B, 768]

    # ── Forward (train) ───────────────────────────────────────────
    def forward(self, pr_ids, pr_mask, r_ids, r_mask):
        p = self.encode(pr_ids,  pr_mask)     # [B, 768]
        r = self.encode(r_ids,   r_mask)      # [B, 768]

        # cosine similarity (for inference / calibration)
        cos_sim = F.cosine_similarity(p, r, dim=1, eps=1e-8)   # [B]

        # rich feature vector for classification head
        feat   = torch.cat([p, r, torch.abs(p - r), p * r], dim=1)  # [B, 3072]
        logits = self.classifier(feat)         # [B, 3]

        return logits, cos_sim, p, r

    # ── Cosine similarity only (inference) ────────────────────────
    @torch.no_grad()
    def similarity(self, pr_ids, pr_mask, r_ids, r_mask):
        p = self.encode(pr_ids,  pr_mask)
        r = self.encode(r_ids,   r_mask)
        return F.cosine_similarity(p, r, dim=1, eps=1e-8)


### Training & Evaluation Functions

In [ ]:
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimiser, scheduler, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0

    for batch in tqdm(loader, desc="  Train", leave=False):
        pr_ids   = batch["pr_ids"].to(device)
        pr_mask  = batch["pr_mask"].to(device)
        r_ids    = batch["r_ids"].to(device)
        r_mask   = batch["r_mask"].to(device)
        labels   = batch["label"].to(device)

        optimiser.zero_grad()
        logits, _, _, _ = model(pr_ids, pr_mask, r_ids, r_mask)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        n          += labels.size(0)

    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, device, return_sims=False):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    all_preds, all_labels, all_sims = [], [], []

    for batch in tqdm(loader, desc="  Eval ", leave=False):
        pr_ids  = batch["pr_ids"].to(device)
        pr_mask = batch["pr_mask"].to(device)
        r_ids   = batch["r_ids"].to(device)
        r_mask  = batch["r_mask"].to(device)
        labels  = batch["label"].to(device)

        logits, cos_sim, _, _ = model(pr_ids, pr_mask, r_ids, r_mask)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        preds       = logits.argmax(1)
        correct    += (preds == labels).sum().item()
        n          += labels.size(0)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        if return_sims:
            all_sims.extend(cos_sim.cpu().tolist())

    avg_loss = total_loss / n
    acc      = correct / n
    wf1      = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

    if return_sims:
        return avg_loss, acc, wf1, all_preds, all_labels, np.array(all_sims)
    return avg_loss, acc, wf1, all_preds, all_labels


### Run Fine-Tuning

In [ ]:
tokenizer = BertTokenizer.from_pretrained(cfg.BERT_MODEL)
model     = QualCheckModel(cfg.BERT_MODEL, cfg.NUM_LABELS, cfg.DROPOUT).to(cfg.DEVICE)

train_ds = QualCheckDataset(train_df, tokenizer, cfg.MAX_LEN)
val_ds   = QualCheckDataset(val_df,   tokenizer, cfg.MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

total_steps   = len(train_loader) * cfg.MAX_EPOCHS
warmup_steps  = int(total_steps * cfg.WARMUP_RATIO)

optimiser  = AdamW(model.parameters(), lr=cfg.LR, eps=1e-8, weight_decay=0.01)
scheduler  = get_linear_schedule_with_warmup(optimiser, warmup_steps, total_steps)

# ── Training loop ──────────────────────────────────────────────────
history = {k: [] for k in ["tr_loss","tr_acc","val_loss","val_acc","val_wf1"]}
best_val_wf1 = 0.0
no_improve   = 0

print(f"Training on {cfg.DEVICE}  |  {total_steps} total steps")
print("="*60)

for epoch in range(cfg.MAX_EPOCHS):
    tr_loss, tr_acc            = train_one_epoch(model, train_loader, optimiser, scheduler, cfg.DEVICE)
    val_loss, val_acc, val_wf1, _, _ = evaluate(model, val_loader, cfg.DEVICE)

    history["tr_loss"].append(tr_loss);  history["tr_acc"].append(tr_acc)
    history["val_loss"].append(val_loss); history["val_acc"].append(val_acc)
    history["val_wf1"].append(val_wf1)

    marker = ""
    if val_wf1 > best_val_wf1:
        best_val_wf1 = val_wf1
        torch.save(model.state_dict(), cfg.BEST_CKPT)
        marker = "  ← best saved"
        no_improve = 0
    else:
        no_improve += 1

    print(f"Epoch {epoch+1}/{cfg.MAX_EPOCHS}  tr_loss={tr_loss:.4f}  tr_acc={tr_acc:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  val_wf1={val_wf1:.4f}{marker}")

    if no_improve >= cfg.PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\nBest Val Weighted-F1 : {best_val_wf1:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs_ran = range(1, len(history["tr_loss"]) + 1)

axes[0].plot(epochs_ran, history["tr_loss"],  label="Train")
axes[0].plot(epochs_ran, history["val_loss"], label="Val")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(epochs_ran, history["tr_acc"],  label="Train")
axes[1].plot(epochs_ran, history["val_acc"], label="Val")
axes[1].set_title("Accuracy"); axes[1].legend()

axes[2].plot(epochs_ran, history["val_wf1"], color="green", marker="o")
axes[2].axhline(0.80, color="red", linestyle="--", label="Threshold 0.80")
axes[2].set_title("Val Weighted F1"); axes[2].legend()

for ax in axes:
    ax.set_xlabel("Epoch")
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle("QualCheck Fine-Tuning History", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{cfg.OUTPUT_DIR}/training_history.png", dpi=150, bbox_inches="tight")
plt.show()


## Phase 3 – Cosine Similarity Scoring

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(cfg.BEST_CKPT, map_location=cfg.DEVICE))
model.eval()

@torch.no_grad()
def extract_cosine_sims(model, dataframe, tokenizer, cfg):
    """Returns (cos_sims, true_labels) for a split."""
    ds     = QualCheckDataset(dataframe, tokenizer, cfg.MAX_LEN)
    loader = DataLoader(ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

    sims, labels = [], []
    for batch in tqdm(loader, desc="  Embedding", leave=False):
        cos = model.similarity(
            batch["pr_ids"].to(cfg.DEVICE),  batch["pr_mask"].to(cfg.DEVICE),
            batch["r_ids"].to(cfg.DEVICE),   batch["r_mask"].to(cfg.DEVICE)
        )
        sims.extend(cos.cpu().tolist())
        labels.extend(batch["label"].tolist())
    return np.array(sims), np.array(labels)

print("Extracting cosine similarities …")
val_sims,  val_true  = extract_cosine_sims(model, val_df,  tokenizer, cfg)
test_sims, test_true = extract_cosine_sims(model, test_df, tokenizer, cfg)

print(f"Val  sims  –  min:{val_sims.min():.4f}  max:{val_sims.max():.4f}  mean:{val_sims.mean():.4f}")
print(f"Test sims  –  min:{test_sims.min():.4f}  max:{test_sims.max():.4f}  mean:{test_sims.mean():.4f}")


In [ ]:
label_names = ["Irrelevant", "Partially Relevant", "Fully Relevant"]
colors      = ["#e74c3c",   "#f39c12",             "#27ae60"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lid in range(3):
    mask = val_true == lid
    axes[0].hist(val_sims[mask], bins=30, alpha=0.6,
                 label=label_names[lid], color=colors[lid])
axes[0].set_title("Cosine Similarity Distribution by Class (Val)")
axes[0].set_xlabel("Cosine Similarity  s"); axes[0].set_ylabel("Count")
axes[0].legend()

data_by_class = [val_sims[val_true == i] for i in range(3)]
bp = axes[1].boxplot(data_by_class, labels=label_names, patch_artist=True, notch=False)
for patch, c in zip(bp["boxes"], colors):
    patch.set_facecolor(c); patch.set_alpha(0.7)
axes[1].set_title("Cosine Similarity Boxplot by Class (Val)")
axes[1].set_ylabel("Cosine Similarity  s")

plt.suptitle("Semantic Alignment Score Distribution", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{cfg.OUTPUT_DIR}/similarity_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


## Phase 4 – Threshold Calibration (Grid Search)

Equation 3.3 from the thesis:

$$C(s) = \begin{cases} \text{Fully Relevant} & s \ge \theta_1 \\ \text{Partially Relevant} & \theta_2 \le s < \theta_1 \\ \text{Irrelevant} & s < \theta_2 \end{cases}$$

In [ ]:
def classify(sims, theta1, theta2):
    """Apply rubric-based 3-class classification C(s)."""
    preds = np.full(len(sims), 0, dtype=int)
    preds[sims >= theta1]                              = 2  # Fully Relevant
    preds[(sims >= theta2) & (sims < theta1)]          = 1  # Partially Relevant
    # sims < theta2                                    = 0  # Irrelevant
    return preds


def grid_search_thresholds(sims, true_labels, step=0.01):
    """Exhaustive grid search – maximises (accuracy + weighted-F1) / 2."""
    candidates = np.arange(float(sims.min()), float(sims.max()), step)
    best = dict(theta1=0.8, theta2=0.5, score=0.0, acc=0.0, wf1=0.0)
    records = []

    for t1 in candidates:
        for t2 in candidates:
            if t2 >= t1:
                continue
            preds = classify(sims, t1, t2)
            acc  = accuracy_score(true_labels, preds)
            wf1  = f1_score(true_labels, preds, average="weighted", zero_division=0)
            combined = (acc + wf1) / 2.0
            records.append({"theta1": round(t1,4), "theta2": round(t2,4),
                             "accuracy": acc, "weighted_f1": wf1, "combined": combined})
            if combined > best["score"]:
                best.update(theta1=t1, theta2=t2, score=combined, acc=acc, wf1=wf1)

    return best, pd.DataFrame(records)


print("Running grid search on validation set …  (step = 0.01)")
best_thresh, thresh_df = grid_search_thresholds(val_sims, val_true, cfg.THRESH_STEP)

θ1 = best_thresh["theta1"]
θ2 = best_thresh["theta2"]

print(f"\n{'='*50}")
print(f" Optimal thresholds")
print(f"  θ₁  (Fully Relevant   boundary) : {θ1:.4f}")
print(f"  θ₂  (Irrelevant       boundary) : {θ2:.4f}")
print(f"  Combined score (acc+wf1)/2      : {best_thresh['score']:.4f}")
print(f"  Val accuracy                    : {best_thresh['acc']:.4f}")
print(f"  Val weighted-F1                 : {best_thresh['wf1']:.4f}")
print(f"{'='*50}")
print(f"\nClassification rule:")
print(f"  s ≥ {θ1:.4f}                 → Fully Relevant")
print(f"  {θ2:.4f} ≤ s < {θ1:.4f}  → Partially Relevant")
print(f"  s <  {θ2:.4f}                 → Irrelevant")

print("\nTop 10 threshold pairs:")
print(thresh_df.nlargest(10, "combined")[["theta1","theta2","accuracy","weighted_f1","combined"]]      .to_string(index=False))


## Phase 5 – System Evaluation

### Classification Performance (Test Set)

In [ ]:
qc_preds = classify(test_sims, θ1, θ2)

acc    = accuracy_score(test_true, qc_preds)
prec   = precision_score(test_true, qc_preds, average="weighted", zero_division=0)
rec    = recall_score(test_true, qc_preds, average="weighted", zero_division=0)
f1_mac = f1_score(test_true, qc_preds, average="macro",    zero_division=0)
f1_w   = f1_score(test_true, qc_preds, average="weighted", zero_division=0)

print("QualCheck – Test Set Results")
print("-"*40)
print(f"Overall Accuracy       : {acc:.4f}")
print(f"Precision  (weighted)  : {prec:.4f}")
print(f"Recall     (weighted)  : {rec:.4f}")
print(f"F1-Score   (macro avg) : {f1_mac:.4f}")
print(f"Weighted F1-Score      : {f1_w:.4f}")
viability = "✓ PASSED" if f1_w >= 0.80 else "✗ FAILED"
print(f"\nViability threshold (≥ 0.80) : {viability}")

print("\nDetailed Classification Report:")
print(classification_report(test_true, qc_preds,
      target_names=["Irrelevant","Partially Relevant","Fully Relevant"]))


In [ ]:
cm = confusion_matrix(test_true, qc_preds)
cls_names = ["Irrelevant", "Partially\nRelevant", "Fully\nRelevant"]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=cls_names, yticklabels=cls_names,
            linewidths=0.5, linecolor="grey")
ax.set_title("QualCheck – Confusion Matrix (Test Set)", fontweight="bold", pad=12)
ax.set_ylabel("True Label"); ax.set_xlabel("Predicted Label")
plt.tight_layout()
plt.savefig(f"{cfg.OUTPUT_DIR}/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


### Semantic Alignment Scoring Quality (MAE, RMSE, Pearson r)

In [ ]:
# Normalise ordinal labels to [0, 1]  :  0→0.0, 1→0.5, 2→1.0
norm_true = test_true / (cfg.NUM_LABELS - 1)

mae  = np.mean(np.abs(test_sims - norm_true))
rmse = np.sqrt(np.mean((test_sims - norm_true) ** 2))
r, p_r = pearsonr(test_sims, norm_true)

print("Semantic Alignment Scoring Evaluation")
print("-"*40)
print(f"MAE          : {mae:.4f}")
print(f"RMSE         : {rmse:.4f}")
print(f"Pearson  r   : {r:.4f}  (p = {p_r:.4f})")
pearson_ok = "✓ PASSED" if r >= 0.70 else "✗ FAILED"
print(f"Threshold (r ≥ 0.70) : {pearson_ok}")


### Baseline Comparisons

In [ ]:
# ── Baseline 1 : TF-IDF Cosine Similarity ─────────────────────────
print("--- Baseline 1: TF-IDF Cosine Similarity ---")

tfidf_all = (list(train_df["tfidf_question"] + " " + train_df["tfidf_rubric"]) +
             list(train_df["tfidf_response"]))
tfidf_vec = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2))
tfidf_vec.fit(tfidf_all)

def tfidf_sims_for(df_split):
    pr_vecs = tfidf_vec.transform(df_split["tfidf_question"] + " " + df_split["tfidf_rubric"])
    r_vecs  = tfidf_vec.transform(df_split["tfidf_response"])
    return np.array([tfidf_cos(pr_vecs[i], r_vecs[i])[0, 0]
                     for i in range(len(df_split))])

bl1_val_sims  = tfidf_sims_for(val_df)
bl1_test_sims = tfidf_sims_for(test_df)

bl1_best, _   = grid_search_thresholds(bl1_val_sims, val_true, 0.01)
bl1_preds     = classify(bl1_test_sims, bl1_best["theta1"], bl1_best["theta2"])
bl1_acc       = accuracy_score(test_true, bl1_preds)
bl1_wf1       = f1_score(test_true, bl1_preds, average="weighted", zero_division=0)
bl1_r, _      = pearsonr(bl1_test_sims, norm_true)
bl1_mae       = np.mean(np.abs(bl1_test_sims - norm_true))

print(f"  Accuracy     : {bl1_acc:.4f}")
print(f"  Weighted F1  : {bl1_wf1:.4f}")
print(f"  Pearson r    : {bl1_r:.4f}")
print(f"  MAE          : {bl1_mae:.4f}")


In [ ]:
# ── Baseline 2 : BERT Cosine Similarity without rubric ─────────────
print("--- Baseline 2: BERT Cosine Similarity (no rubric) ---")

@torch.no_grad()
def bert_only_sims(model, dataframe, tokenizer, cfg):
    """Encode question-only (no rubric) vs response."""
    sims = []
    model.eval()
    for _, row in tqdm(dataframe.iterrows(), total=len(dataframe),
                       desc="  BL2", leave=False):
        q_enc = tokenizer.encode_plus(
            row["clean_question"],
            add_special_tokens=True, max_length=cfg.MAX_LEN,
            padding="max_length", truncation=True,
            return_attention_mask=True, return_tensors="pt")
        r_enc = tokenizer.encode_plus(
            row["clean_response"],
            add_special_tokens=True, max_length=cfg.MAX_LEN,
            padding="max_length", truncation=True,
            return_attention_mask=True, return_tensors="pt")
        cos = model.similarity(
            q_enc["input_ids"].to(cfg.DEVICE), q_enc["attention_mask"].to(cfg.DEVICE),
            r_enc["input_ids"].to(cfg.DEVICE), r_enc["attention_mask"].to(cfg.DEVICE))
        sims.append(cos.item())
    return np.array(sims)

bl2_val_sims  = bert_only_sims(model, val_df,  tokenizer, cfg)
bl2_test_sims = bert_only_sims(model, test_df, tokenizer, cfg)

bl2_best, _   = grid_search_thresholds(bl2_val_sims, val_true, 0.01)
bl2_preds     = classify(bl2_test_sims, bl2_best["theta1"], bl2_best["theta2"])
bl2_acc       = accuracy_score(test_true, bl2_preds)
bl2_wf1       = f1_score(test_true, bl2_preds, average="weighted", zero_division=0)
bl2_r, _      = pearsonr(bl2_test_sims, norm_true)
bl2_mae       = np.mean(np.abs(bl2_test_sims - norm_true))

print(f"  Accuracy     : {bl2_acc:.4f}")
print(f"  Weighted F1  : {bl2_wf1:.4f}")
print(f"  Pearson r    : {bl2_r:.4f}")
print(f"  MAE          : {bl2_mae:.4f}")


In [ ]:
results = pd.DataFrame({
    "System"       : ["BL1 – TF-IDF", "BL2 – BERT (no rubric)", "QualCheck (Ours)"],
    "Accuracy"     : [bl1_acc,  bl2_acc,  acc],
    "Weighted F1"  : [bl1_wf1,  bl2_wf1,  f1_w],
    "Pearson r"    : [bl1_r,    bl2_r,    r],
    "MAE"          : [bl1_mae,  bl2_mae,  mae],
    "ΔMAE vs BL1"  : [0.0, round(bl2_mae - bl1_mae, 4), round(mae - bl1_mae, 4)],
    "ΔWF1 vs BL1"  : [0.0, round(bl2_wf1 - bl1_wf1, 4), round(f1_w - bl1_wf1, 4)],
})
results.to_csv(f"{cfg.OUTPUT_DIR}/results_summary.csv", index=False)

print("Results Summary")
print(results.to_string(index=False))

# ── Comparison bar chart ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sys_labels = ["BL1\n(TF-IDF)", "BL2\n(BERT)", "QualCheck"]
bar_colors = ["#bdc3c7", "#f0b27a", "#82e0aa"]

for ax, metric in zip(axes, ["Weighted F1", "Accuracy"]):
    vals = results[metric].tolist()
    bars = ax.bar(sys_labels, vals, color=bar_colors, edgecolor="black", width=0.55)
    ax.set_ylim(0, 1.1); ax.set_ylabel(metric)
    ax.set_title(f"{metric} Comparison", fontweight="bold")
    ax.axhline(0.80, color="red", linestyle="--", lw=1.2, label="Threshold 0.80")
    ax.legend(fontsize=9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{v:.4f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.suptitle("System Comparison – Test Set", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{cfg.OUTPUT_DIR}/system_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


### Statistical Tests

In [ ]:
print("1. Score-Level Agreement (Pearson r)")
print(f"   QualCheck :  r = {r:.4f},  p = {p_r:.4f}")
print(f"   Target r ≥ 0.70 : {'✓ PASSED' if r >= 0.70 else '✗ FAILED'}\n")

# ── Descriptive statistics by class ───────────────────────────────
print("2. Descriptive Statistics – Cosine Similarity by Relevance Class")
print(f"{'Class':<22} {'Mean':>6} {'Median':>8} {'Std':>6} {'IQR':>6}")
print("-" * 52)
for lid, lname in cfg.ID2LABEL.items():
    mask = test_true == lid
    if mask.any():
        s = test_sims[mask]
        iqr = np.percentile(s, 75) - np.percentile(s, 25)
        print(f"{lname:<22} {s.mean():>6.4f} {np.median(s):>8.4f} {s.std():>6.4f} {iqr:>6.4f}")

# ── One-Way ANOVA (if output_type column exists) ───────────────────
type_col = cfg.COL_OUTPUT_TYPE
if type_col and type_col in test_df.columns:
    print(f"\n3. One-Way ANOVA – Cross-Output-Type Consistency")
    groups     = test_df[type_col].unique()
    group_sims = [test_sims[test_df[type_col].values == g] for g in groups]
    F, p_anova = f_oneway(*group_sims)
    print(f"   F = {F:.4f},  p = {p_anova:.4f}")

    if p_anova > 0.05:
        print("   Non-significant (p > 0.05): consistent across output types ✓")
    else:
        print("   Significant (p ≤ 0.05): Tukey HSD post-hoc analysis:")
        all_s = np.concatenate(group_sims)
        all_g = np.concatenate([[g]*len(s) for g, s in zip(groups, group_sims)])
        print(pairwise_tukeyhsd(all_s, all_g, alpha=0.05))
else:
    print("\n3. One-Way ANOVA : skipped (no output_type column found)")
    print("   Add a column named 'output_type' with values ")
    print("   ('short_answer', 'essay', 'code_report') to enable this test.")


## Save Final Model & Predictions

In [ ]:
# ── Save full checkpoint (model + thresholds) ─────────────────────
torch.save({
    "model_state_dict" : model.state_dict(),
    "theta1"           : θ1,
    "theta2"           : θ2,
    "bert_model"       : cfg.BERT_MODEL,
    "max_len"          : cfg.MAX_LEN,
    "label_map"        : cfg.LABEL2ID,
}, cfg.FINAL_MODEL)

# ── Save test predictions ─────────────────────────────────────────
out_df = test_df[[cfg.COL_QUESTION, cfg.COL_RUBRIC, cfg.COL_RESPONSE,
                  cfg.COL_LABEL]].copy()
out_df["cosine_similarity"] = test_sims
out_df["predicted_label"]   = [cfg.ID2LABEL[p] for p in qc_preds]
out_df["correct"]           = (test_true == qc_preds)
out_df.to_csv(f"{cfg.OUTPUT_DIR}/test_predictions.csv", index=False)

# ── Print final summary ───────────────────────────────────────────
print("="*60)
print(" QUALCHECK – FINAL RESULTS")
print("="*60)
print(f" θ₁ = {θ1:.4f}   θ₂ = {θ2:.4f}")
print(f" Accuracy       : {acc:.4f}")
print(f" Weighted F1    : {f1_w:.4f}  ({'✓' if f1_w>=0.80 else '✗'} viability ≥ 0.80)")
print(f" Pearson r      : {r:.4f}  ({'✓' if r>=0.70 else '✗'} target ≥ 0.70)")
print(f" MAE            : {mae:.4f}")
print(f" RMSE           : {rmse:.4f}")
print("-"*60)
print(f" ΔWF1 vs BL1 (TF-IDF)        : {f1_w - bl1_wf1:+.4f}")
print(f" ΔWF1 vs BL2 (BERT-no-rubric): {f1_w - bl2_wf1:+.4f}")
print("="*60)

print("\nSaved files:")
for f in [cfg.BEST_CKPT, cfg.FINAL_MODEL,
          f"{cfg.OUTPUT_DIR}/test_predictions.csv",
          f"{cfg.OUTPUT_DIR}/results_summary.csv",
          f"{cfg.OUTPUT_DIR}/training_history.png",
          f"{cfg.OUTPUT_DIR}/similarity_distribution.png",
          f"{cfg.OUTPUT_DIR}/confusion_matrix.png",
          f"{cfg.OUTPUT_DIR}/system_comparison.png"]:
    print(f"  {f}")
print("\n✓ QualCheck pipeline complete!")
